# Blif Generator

This code generate blif files using yosys and ABC.

In [1]:
import os
from jinja2 import Template

In [2]:
N    = 5
PATH = f'{os.getcwd()}/results/blif_generator'
# Verify if the directory exists
if not os.path.exists(PATH):
    os.makedirs(PATH)





## Generate .pla file

In [3]:
def generate_pla(n):
    """
    Gera a tabela verdade no formato PLA para um somador de dois números de n bits.

    :param n: Número de bits de cada número de entrada.
    """
    num_inputs = 2 * n  # Número total de bits de entrada (dois números de n bits)
    num_outputs = n + 1  # Número total de bits de saída (n bits + carry out)

    # Cabeçalho do formato PLA
    pla = [f".i {num_inputs}", f".o {num_outputs}"]

    # Itera sobre todas as combinações de entrada
    for i in range(2 ** num_inputs):
        # Converte o índice para uma sequência binária de entrada
        inputs = f"{i:0{num_inputs}b}"

        # Divide os bits em dois números
        a = int(inputs[:n], 2)
        b = int(inputs[n:], 2)

        # Calcula a soma dos dois números
        total = a + b

        # Converte o resultado para uma sequência binária de saída
        outputs = f"{total:0{num_outputs}b}"

        # Adiciona a linha de entrada e saída ao PLA
        pla.append(f"{inputs} {outputs}")

    # Retorna o PLA como uma string
    pla.append(".e")
    return "\n".join(pla)

# Número de bits dos números de entrada
pla_output = generate_pla(N)

# Salva o resultado em um arquivo
with open(f"{PATH}/somador.pla", "w") as file:
    file.write(pla_output)


## Generate .blif file using yosys



In [4]:

# Template do arquivo ABC
with open(f"/app/templates/read_pla.jinja") as file:
    template_yosys = Template(file.read())

with open(f"/app/templates/abc_sript.jinja") as file:
    template_abc = Template(file.read())

# Renderiza o template com o PLA gerado
abc_script = template_abc.render(pla_file=f"{PATH}/somador.pla",
                                otimizations= ["resyn3", 'resyn'],
                                lut_k = 5,
                                verilog_output=f"{PATH}/somador.v",
                                dot_output=f"{PATH}/somador.dot",
                                blif_output=f"{PATH}/somador.blif")
                                
with open(f"{PATH}/abc_script.rc", "w") as file:
    file.write(abc_script)

yosys_script = template_yosys.render(pla=f"{PATH}/somador.pla",
                                    abc_script=f"{PATH}/abc_script.abc",
                                    verilog_output=f"{PATH}/somador.v",
                                    dot_output=f"{PATH}/somador.dot",
                                    blif_output=f"{PATH}/somador.blif")

with open(f"{PATH}/yosys_script.ys", "w") as file:
    file.write(yosys_script)

# Executa o script do Yosys

os.system(f"yosys-abc -f {PATH}/abc_script.rc")






ABC command line: "source /app/results/blif_generator/abc_script.rc".

/app/results/blif_generator/somador: i/o =   10/    6  lat =    0  and =    357  lev = 18
/app/results/blif_generator/somador: i/o =   10/    6  lat =    0  and =    325  lev = 17
/app/results/blif_generator/somador: i/o =   10/    6  lat =    0  nd =    78  edge =    343  aig  =   477  lev = 5
/app/results/blif_generator/somador: i/o =   10/    6  lat =    0  nd =    78  edge =    343  bdd  =   520  lev = 5
Limpeza do circuito concluída 


0

## Evalutation

In [5]:
def gerar_testbench(n):
    """
    Gera um testbench Verilog parametrizado para testar o módulo `somador`.

    Parâmetros:
        n (int): Número de bits de cada operando (A e B).

    Retorna:
        str: Código Verilog do testbench.
    """
    # Verificar se o valor de n é válido
    if n < 1:
        raise ValueError("O número de bits deve ser pelo menos 1.")

    # Identificar as variáveis de entrada e saída
    inputs = [f"x{i}" for i in range(2 * n)]
    outputs = [f"z{i}" for i in range(n + 1)]

    # Cabeçalho do testbench
    testbench = f"""`include "{PATH}/somador.v"

module tb_somador;

  // Entradas
  reg {', '.join(inputs)};

  // Saídas
  wire {', '.join(outputs)};

  // Instância do módulo a ser testado
  \/app/results/blif_generator/somador uut (
    .{', .'.join([f'{inp}({inp})' for inp in inputs + outputs])}
  );

  integer A, B, C;

  initial begin
    $display(\"Iniciando os testes...\");

    // Testando todas as combinações de A e B
    for (A = 0; A < {1 << n}; A = A + 1) begin
      for (B = 0; B < {1 << n}; B = B + 1) begin
        // Configurar os bits de entrada
"""

    # Adicionar as atribuições para cada bit de A e B
    for i in range(n):
        testbench += f"        x{i} = A[{n - 1 - i}];\n"
    for i in range(n):
        testbench += f"        x{n + i} = B[{n - 1 - i}];\n"

    # Continuar o restante do código
    testbench += f"""
        // Aguardar propagação
        #10;

        // Calcular a soma esperada
        C = A + B;

        // Verificar o resultado
        if (C !== {{ {', '.join((outputs))} }}) begin
          $display(\"Erro: A=%b, B=%b -> Esperado=%b, Obtido=%b\", A, B, C, {{ {', '.join(reversed(outputs))} }});
        end
      end
    end

    $display(\"Testes concluídos.\");
    $finish;
  end
endmodule
"""

    return testbench

# Exemplo de uso
codigo_testbench = gerar_testbench(N)

# Salvar o testbench em um arquivo
with open(f"{PATH}/tb_somador.v", "w") as arquivo:
    arquivo.write(codigo_testbench)

print("Testbench gerado com sucesso!")


Testbench gerado com sucesso!


<>:44: SyntaxWarning: invalid escape sequence '\/'
<>:44: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_100979/194710385.py:44: SyntaxWarning: invalid escape sequence '\/'
  """


In [6]:
# Run the testbench
os.system(f"iverilog -o {PATH}/tb_somador.vvp {PATH}/tb_somador.v")
# Get time of execution
os.system(f"vvp {PATH}/tb_somador.vvp")

Iniciando os testes...
Testes concluídos.


0